# Bài 3. Minimax và cắt tỉa Alpha-Beta

## Introduction 

Multiplayer games can be implemented as:
1. Nondeterministic actions: The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. __Optimal Decisions:__ Minimax search (search complete game tree) and alpha-beta pruning.
3. Heuristic Alpha-Beta Tree Search: Cut off tree search and use heuristic to estimate state value. 
4. Monte Carlo Tree search: Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)). The game is a __zero-sum game__: Win by x results in +1, win by o in -1 and a tie has a value of 0. Max plays x and tries to maximize the outcome while Min plays o and tries to minimize the outcome.   

We will implement
* __minimax search,__
* __minimax search with alpha-beta pruning,__ and
* __move ordering__. 

The search time (number if nodes explored) could be further improved by 
* __heuristic alpha-beta tree search__ (cut off search and use heuristic evaluation function to approximate utility), 
* __forward pruning__ (prune moves that appear poor), and
* __table lookups__ (for openings and end game).

The algorithms search the game tree and we could return a conditional plan (or partial plan if cut offs are used), but the implementation here only identifies and returns the optimal next move.

Các hàm biểu diễn và chơi Tic-Tac-Toe được nhập từ [tictactoe.py](tictactoe.py) trong cùng thư mục.

In [1]:
from time import perf_counter

import numpy as np
import pandas as pd

from tictactoe import actions, empty_board, other, play, random_player, result, utility

# Recursive DFS algorithm for Minimax Search

See AIMA page 150. 

Note that the minimax search algorithm is very similar to AND-OR search where max does OR and min is represented by AND nodes.

In [2]:
import math

# global variables
DEBUG = 0 # 1 ... count nodes, 2 ... debug each node
COUNT = 0

def minimax_search(board, player = 'x'):
    """start the search."""
    global DEBUG, COUNT
    COUNT = 0
    
    value, move = max_value(board, player)
    
    if DEBUG >= 1: print(f"Number of nodes searched: {COUNT}") 
 
    return { "move": move, "value": value}

def max_value(state, player):
    """player's best move."""
    global DEBUG, COUNT
    COUNT += 1
    
    # return utility of state if it is a terminal state
    v = utility(state, player)
    if DEBUG >= 2: print("max in: " + str(state) + str([v]) ) 
    if v is not None: return v, None
        
    v, move = -math.inf, None

    # check all possible actions in the state, return move with the largest value
    for a in actions(state):
        v2, a2 = min_value(result(state, player, a), player)
        if v2 > v:
            v, move = v2, a
    
    if DEBUG >= 2: print("max out: " + str(state) + str([v, move]) ) 
    return v, move

def min_value(state, player):
    """opponent's best response."""
    global DEBUG, COUNT
    COUNT += 1
    
    # return utility of state if it is a terminal state
    v = utility(state, player)
    if DEBUG >= 2: print("min in: " + str(state) + str([v]) ) 
    if v is not None: return v, None
    
    v, move = +math.inf, None

    # check all possible actions in the state, return move with the smallest value
    for a in actions(state):
        v2, a2 = max_value(result(state, other(player), a), player)
        if v2 < v:
            v, move = v2, a
    
    if DEBUG >= 2: print("min out: " + str(state) + str([v, move]) ) 
    return v, move

## Minimax với cắt tỉa Alpha-Beta

Xét theo góc nhìn của người sắp đi, Max tìm giá trị lớn nhất và Min tìm giá trị nhỏ nhất. Alpha là cận dưới mà Max đã bảo đảm, còn beta là cận trên mà Min đã bảo đảm. Khi một nhánh không thể cải thiện lựa chọn đã có ở phía trên, thuật toán bỏ qua những nước còn lại của nhánh đó.

In [3]:
def alpha_beta_search(board, player='x', *, action_order=actions):
    """Find an optimal move using the given order of legal actions."""
    global DEBUG, COUNT
    COUNT = 0
    
    value, move = max_value_ab(board, player, -math.inf, +math.inf, action_order)
    
    if DEBUG >= 1: print(f"Number of nodes searched: {COUNT}") 
    
    return { "move": move, "value": value }

def max_value_ab(state, player, alpha, beta, action_order):
    """player's best move."""
    global DEBUG, COUNT
    COUNT += 1
       
    # return utility of state is a terminal state
    v = utility(state, player)
    if DEBUG >= 2: print(f"max: {state} [alpha,beta]=[{alpha},{beta}] v={v}")
    if v is not None: 
        if DEBUG >= 2: print(f"     found terminal state. backtracking.")
        return v, None
        
    v, move = -math.inf, None

    # check all possible actions in the state, update alpha and return move with the largest value
    for a in action_order(state):
        v2, a2 = min_value_ab(result(state, player, a), player, alpha, beta, action_order)
        if DEBUG >= 2: print(f"max: {state} (backtracked) [alpha,beta]=[{alpha},{beta}] v={v2}")
        
        if v2 > v:
            v, move = v2, a
            alpha = max(alpha, v)
        if v >= beta:
            if DEBUG >= 2: print(f"     v>=beta ({v}>={beta}): pruning remaining subtree (actions). backtracking.")
            return v, move
    
    return v, move

def min_value_ab(state, player, alpha, beta, action_order):
    """opponent's best response."""
    global DEBUG, COUNT
    COUNT += 1
    
    # return utility of state is a terminal state
    v = utility(state, player)
    if DEBUG >= 2: print(f"min: {state} [alpha,beta]=[{alpha},{beta}] v={v}")
    if v is not None: 
        if DEBUG >= 2: print(f"     found terminal state. backtracking.")
        return v, None
    
    v, move = +math.inf, None

    # check all possible actions in the state, update beta and return move with the smallest value
    for a in action_order(state):
        v2, a2 = max_value_ab(result(state, other(player), a), player, alpha, beta, action_order)
        if DEBUG >= 2: print(f"min: {state} (backtracked) [alpha,beta]=[{alpha},{beta}] v={v2}")
        
        if v2 < v:
            v, move = v2, a
            beta = min(beta, v)
        if v <= alpha: 
            if DEBUG >= 2: print(f"     v<=alpha ({v}<={alpha}): pruning remaining subtree (actions). backtracking.")
            return v, move
    
    return v, move

## Thí nghiệm 1 · So sánh trên cùng bốn bàn cờ

Cả hai thuật toán đều tìm nước cho X và duyệt các ô trống theo thứ tự chỉ số từ 0 đến 8. Mỗi bàn dưới đây ghi ba hàng từ trên xuống. Dấu chấm là ô trống.

| Tình huống | Bàn cờ | Giá trị kỳ vọng cho X |
| --- | --- | ---: |
| X có đường thắng | XO. / OX. / ... | +1 |
| X có thể giữ hòa | XOX / .O. / ... | 0 |
| X ở thế thua | O.X / ... / ..O | -1 |
| Bàn rỗng | ... / ... / ... | 0 |

Mỗi lần tìm kiếm tự đặt lại bộ đếm. Bảng ghi số lần thăm nút, gồm cả nút gốc, và thời gian chạy tìm kiếm đo bằng `perf_counter`. Thời gian chỉ phản ánh lần đo này và không gồm kiểm tra hay hiển thị bảng.

In [4]:
test_cases = [
    ("X có đường thắng", list("xo ox    "), 1),
    ("X có thể giữ hòa", list("xox o    "), 0),
    ("X ở thế thua", list("o x     o"), -1),
    ("Bàn rỗng", empty_board(), 0),
]

def measure_search(search, board, **kwargs):
    start = perf_counter()
    answer = search(board, player="x", **kwargs)
    elapsed_ms = (perf_counter() - start) * 1000
    return {
        "Nước đi": answer["move"],
        "Giá trị Minimax": answer["value"],
        "Số nút": COUNT,
        "Thời gian (ms)": elapsed_ms,
    }

def check_optimal_move(board, measurement, expected_value):
    move = measurement["Nước đi"]
    assert move in actions(board)
    assert measurement["Giá trị Minimax"] == expected_value
    # Sau nước của X, O đi trước trong cây con. Kiểm tra ngoài thời gian đo.
    actual_value, _ = min_value(result(board, "x", move), "x")
    assert actual_value == expected_value

comparison_rows = []
alpha_beta_baseline = {}

for name, board, expected_value in test_cases:
    minimax = measure_search(minimax_search, board)
    alpha_beta = measure_search(alpha_beta_search, board)

    for algorithm, measurement in [("Minimax", minimax), ("Alpha-Beta", alpha_beta)]:
        check_optimal_move(board, measurement, expected_value)
        comparison_rows.append({"Bàn cờ": name, "Thuật toán": algorithm, **measurement})

    assert alpha_beta["Số nút"] <= minimax["Số nút"]
    alpha_beta_baseline[name] = alpha_beta

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({"Thời gian (ms)": 3}))

,Bàn cờ,Thuật toán,Nước đi,Giá trị Minimax,Số nút,Thời gian (ms)
0,X có đường thắng,Minimax,2,1,190,5.991
1,X có đường thắng,Alpha-Beta,2,1,61,1.871
2,X có thể giữ hòa,Minimax,7,0,206,6.154
3,X có thể giữ hòa,Alpha-Beta,7,0,101,3.130
4,X ở thế thua,Minimax,1,-1,941,28.838
5,X ở thế thua,Alpha-Beta,1,-1,238,8.455
6,Bàn rỗng,Minimax,0,0,549946,16198.683
7,Bàn rỗng,Alpha-Beta,0,0,18297,547.354


**Nhận xét.** Cả hai thuật toán cho cùng giá trị trên bốn bàn, lần lượt là thắng, hòa, thua và hòa. Trên bàn đầu, nước 2 vẫn dẫn đến thắng nếu X tiếp tục chơi tối ưu, còn nước 8 thắng ngay. Utility chấm hai kết quả thắng như nhau vì không xét số lượt còn lại. Alpha-Beta duyệt ít nút hơn ở cả bốn bàn. Chênh lệch rõ nhất ở bàn rỗng vì cây còn nhiều nước đi.

## Thí nghiệm 2 · Ảnh hưởng của thứ tự duyệt

Ưu tiên ô giữa, sau đó đến các góc, cuối cùng là các ô cạnh. Trong cùng nhóm, thử ô có chỉ số lớn hơn trước, theo thứ tự mẫu. Hai cấu hình đều dùng cùng thuật toán Alpha-Beta và cùng bốn bàn. Số nút trước khi đổi thứ tự lấy từ bảng ở thí nghiệm 1.

In [5]:
def prioritized_actions(board):
    priority = (1, 0, 1, 0, 2, 0, 1, 0, 1)
    return sorted(actions(board), key=lambda move: (priority[move], move), reverse=True)

ordering_rows = []
for name, board, expected_value in test_cases:
    before = alpha_beta_baseline[name]
    after = measure_search(alpha_beta_search, board, action_order=prioritized_actions)
    check_optimal_move(board, after, expected_value)
    ordering_rows.append({
        "Bàn cờ": name,
        "Nước đi trước": before["Nước đi"],
        "Nước đi sau": after["Nước đi"],
        "Giá trị Minimax": after["Giá trị Minimax"],
        "Số nút trước": before["Số nút"],
        "Số nút sau": after["Số nút"],
        "Thay đổi số nút (%)": 100 * (after["Số nút"] / before["Số nút"] - 1),
    })

display(pd.DataFrame(ordering_rows).round({"Thay đổi số nút (%)": 2}))

,Bàn cờ,Nước đi trước,Nước đi sau,Giá trị Minimax,Số nút trước,Số nút sau,Thay đổi số nút (%)
0,X có đường thắng,2,8,1,61,42,-31.15
1,X có thể giữ hòa,7,7,0,101,106,4.95
2,X ở thế thua,1,4,-1,238,36,-84.87
3,Bàn rỗng,0,4,0,18297,7275,-60.24


**Nhận xét.** Trên bàn rỗng, số nút giảm từ 18.297 xuống 7.275 (60,24%). Thuật toán chọn ô 0 theo thứ tự gốc và ô 4 theo thứ tự ưu tiên. Hai nước đều đạt giá trị 0. Tại nút Max, tìm thấy phương án mạnh sớm làm alpha tăng. Tại nút Min, tìm thấy phản công mạnh sớm làm beta giảm. Khi alpha gặp hoặc vượt beta, các nhánh còn lại không thể cải thiện lựa chọn ở nút phía trên. Quy tắc ưu tiên vị trí chỉ dự đoán thứ tự tốt. Ở bàn X có thể giữ hòa, thứ tự này làm số nút tăng từ 101 lên 106.

**Câu thảo luận 2.** Alpha-Beta vẫn tìm một nước đi tối ưu theo Minimax vì chỉ cắt các nhánh không thể cải thiện quyết định. Khi nhiều nước có cùng giá trị tối ưu, thứ tự duyệt có thể làm thuật toán trả về một nước đồng hạng khác.

## Thí nghiệm 3 · Alpha-Beta đấu Random

Dùng Alpha-Beta có ưu tiên tâm, góc, cạnh. Mỗi cấu hình chơi 100 ván với seed cố định. X luôn đi trước theo luật của hàm `play`. Thời gian trung bình chỉ tính các lượt của Alpha-Beta.

In [6]:
def alpha_beta_player(board, player="x"):
    return alpha_beta_search(board, player, action_order=prioritized_actions)["move"]

def play_against_random(alpha_beta_side, games):
    move_times = []

    def timed_player(board, player):
        start = perf_counter()
        move = alpha_beta_player(board, player)
        move_times.append(perf_counter() - start)
        return move

    if alpha_beta_side == "x":
        scores = play(timed_player, random_player, N=games)
    else:
        scores = play(random_player, timed_player, N=games)

    assert sum(scores.values()) == games
    return {
        "Quân Alpha-Beta": alpha_beta_side.upper(),
        "Số ván": games,
        "Thắng": scores[alpha_beta_side],
        "Hòa": scores["d"],
        "Thua": scores[other(alpha_beta_side)],
        "Thời gian TB/nước (ms)": 1000 * np.mean(move_times),
    }

np.random.seed(25)
match_results = pd.DataFrame([play_against_random(side, games=100) for side in ("x", "o")])
display(match_results.round({"Thời gian TB/nước (ms)": 3}))

,Quân Alpha-Beta,Số ván,Thắng,Hòa,Thua,Thời gian TB/nước (ms)
0,X,100,98,2,0,68.400
1,O,100,91,9,0,18.291


**Nhận xét.** Alpha-Beta thắng 98 ván khi cầm X và 91 ván khi cầm O. Các ván còn lại đều hòa. Việc thắng nhiều ván trước Random cho thấy tác tử khai thác được các nước yếu của đối thủ. Tỷ lệ thắng theo từng vai phụ thuộc vào việc X đi trước và các nước ngẫu nhiên trong mẫu. Khi đối thủ cũng chơi tối ưu, giá trị 0 của bàn rỗng dự đoán một ván hòa.